# TD 3 : statistiques descriptives et comparaisons

Cours de M1 Économie de l'innovation, Université de Picardie Jules Verne

**La question du TD.** Comment résumer des données et comparer des groupes ou des périodes sans conclure trop vite.

| | |
|---|---|
| En séance (1 h, accompagné) | parties 1 à 3 |
| À distance (environ 3 h 30) | parties 4 à 7 |
| Jalon du projet | la base du projet documentée et deux graphiques commentés |
| Ensuite | TD 4 : de la description à la question |

### Comment travailler avec ce notebook

Chaque partie porte un niveau. Avancez à votre rythme.

- **Essentiel** : à faire par tout le monde. Ces parties suffisent pour le projet.
- **Pour aller plus loin** : pour celles et ceux qui sont à l'aise avec Python.
- **Bonus** : des outils plus avancés (régression, probit, téléchargement direct). Les utiliser dans le projet est un plus. Ne pas les utiliser n'est jamais pénalisé.

Une cellule de code s'exécute en cliquant dessus puis en appuyant sur **Maj + Entrée**. Les zones **À vous** sont à compléter : une ligne de code ou quelques phrases. Une erreur n'est pas grave : lisez la dernière ligne du message, corrigez et relancez la cellule.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Base réunie au TD 2, complétée d'une colonne zone (Europe ou Hors Europe)
base = pd.read_csv("donnees/base-innovation-pays-2000-2023.csv")
base.head()

## 1. Résumer une variable (Essentiel)

La moyenne est sensible aux valeurs extrêmes. La médiane partage les observations en deux moitiés. Quand quelques pays pèsent très lourd, les deux mesures divergent.

In [ ]:
b2021 = base[base["annee"] == 2021]
print("Moyenne :", round(b2021["brevets_par_million"].mean(), 1))
print("Médiane :", round(b2021["brevets_par_million"].median(), 1))
b2021["brevets_par_million"].describe()

In [ ]:
plt.hist(b2021["brevets_par_million"].dropna(), bins=10)
plt.title("Brevets par million d'habitant.es en 2021 (19 pays)")
plt.xlabel("Brevets par million d'habitant.es")
plt.ylabel("Nombre de pays")
plt.show()

**À vous.** Expliquez l'écart entre la moyenne et la médiane à l'aide de l'histogramme. Repérez les pays responsables. Dites quelle mesure vous retiendriez pour décrire un pays typique.

*Votre réponse :*

## 2. Comparer des groupes (Essentiel)

On compare l'Europe et les autres pays de la base.

In [ ]:
b2021.groupby("zone")[["rd_pib", "brevets_par_million", "export_haute_techno"]].median()

**À vous.** Décrivez les écarts entre les deux zones. Dites si la zone Hors Europe forme un groupe homogène (regardez la liste des pays) et ce que cela implique pour la comparaison.

*Votre réponse :*

## 3. Comparer des périodes (Essentiel)

In [ ]:
base["periode"] = "autre"
base.loc[base["annee"].between(2000, 2009), "periode"] = "2000-2009"
base.loc[base["annee"].between(2014, 2023), "periode"] = "2014-2023"

periodes = base[base["periode"] != "autre"]
tableau = periodes.pivot_table(index="pays", columns="periode", values="rd_pib", aggfunc="mean")
tableau["evolution"] = tableau["2014-2023"] - tableau["2000-2009"]
tableau.sort_values("evolution", ascending=False).round(2)

**À vous.** Indiquez les trois pays dont l'intensité de R&D a le plus augmenté entre les deux périodes. Proposez une explication pour l'un d'eux à l'aide d'un chapitre du cours.

*Votre réponse :*

## 4. Croiser deux variables (Essentiel, à distance)

On cherche à savoir si les pays qui consacrent le plus à la R&D déposent aussi le plus de brevets.

In [ ]:
plt.scatter(b2021["rd_pib"], b2021["brevets_par_million"])
for _, ligne in b2021.iterrows():
    plt.annotate(ligne["code"], (ligne["rd_pib"], ligne["brevets_par_million"]))
plt.title("Intensité de R&D et brevets en 2021")
plt.xlabel("R&D (% du PIB)")
plt.ylabel("Brevets par million d'habitant.es")
plt.show()

print("Corrélation :", round(b2021["rd_pib"].corr(b2021["brevets_par_million"]), 2))

**À vous.** Décrivez la relation. Repérez les pays qui s'en écartent et proposez une raison (régime d'appropriation, structure sectorielle, propension à breveter, cf. Chapitres 3 et 5).

*Votre réponse :*

## 5. Une échelle logarithmique (Pour aller plus loin)

Quand une variable va de quelques dizaines à plusieurs milliers, une échelle logarithmique rend le graphique lisible. Chaque graduation multiplie la précédente par dix.

In [ ]:
plt.scatter(b2021["rd_pib"], b2021["brevets_par_million"])
plt.yscale("log")
plt.title("Intensité de R&D et brevets en 2021 (échelle logarithmique)")
plt.xlabel("R&D (% du PIB)")
plt.ylabel("Brevets par million d'habitant.es (échelle log)")
plt.show()

## 6. Une matrice de corrélations et un indice base 100 (Pour aller plus loin)

In [ ]:
colonnes = ["rd_pib", "chercheurs_par_million", "pib_par_habitant", "brevets_par_million", "export_haute_techno"]
b2021[colonnes].corr().round(2)

In [ ]:
# Indice base 100 en 2010 : compare des évolutions entre pays de niveaux différents
choix = ["FRA", "DEU", "KOR", "CHN"]
for code in choix:
    d = base[(base["code"] == code) & (base["annee"] >= 2010)].set_index("annee")["brevets_residents"]
    plt.plot(d.index, d / d.loc[2010] * 100, label=code)
plt.title("Demandes de brevets des résident.es (base 100 en 2010)")
plt.ylabel("Indice")
plt.legend()
plt.show()

**À vous.** Expliquez ce que l'indice montre que les niveaux ne montrent pas.

*Votre réponse :*

**Bonus.** Testez si l'écart d'intensité de R&D entre les deux zones en 2021 est statistiquement significatif.

In [ ]:
from scipy import stats
europe = b2021.loc[b2021["zone"] == "Europe", "rd_pib"].dropna()
autres = b2021.loc[b2021["zone"] == "Hors Europe", "rd_pib"].dropna()
print(stats.ttest_ind(europe, autres, equal_var=False))
# Avec si peu de pays, le test a peu de puissance : un écart non significatif n'est pas une absence d'écart

## 7. Pour le projet (Essentiel)

Dans le notebook de votre projet :

1. décrivez votre base (source, période, unité d'observation, nombre d'observations, valeurs manquantes),
2. produisez deux graphiques avec titre, unités et source,
3. commentez chacun en quelques lignes, en reliant ce que vous voyez à un mécanisme du cours.

Déposez le notebook sur Moodle.

---
Données : Banque mondiale, World Development Indicators, téléchargées en novembre 2026. Fichiers dans le dossier `donnees` (description dans `donnees/LISEZMOI.md`).